# ChatAPG – tabelnodes, grote tabellen en citations

**Doel van deze spike:** zichtbaar maken hoe tabelinhoud van Markdown naar een canonieke tabel, traceerbare nodes en leesbare citation-chunks kan gaan.

Deze notebook verandert geen productiecode en maakt geen keuze voor één bronparser. De generieke kern accepteert een canoniek tabelmodel. De eerste adapter hieronder ondersteunt gangbare pipe-Markdown; SharePoint-HTML en Document Intelligence krijgen later eigen dunne adapters naar hetzelfde model.

## Wat deze spike bewijst

1. Rijen, kolommen, lege waarden en bronposities blijven herkenbaar.
2. Tabel-, rij-, cel- en tekstnodes krijgen stabiele, unieke IDs.
3. Grote tabellen worden uitsluitend tussen volledige rijen gesplitst.
4. Iedere chunk herhaalt de kolomkoppen en blijft als Markdown leesbaar in React.
5. Een citation-node kan worden teruggeleid naar precies de chunk die in de frontend getoond moet worden.

In [ ]:
from hashlib import sha1
import json
import re

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

def show_json(value):
    print(json.dumps(value, indent=2, ensure_ascii=False))

## 1. Representatieve Markdown-input

De data hieronder is synthetisch. Ze bevat percentages, tekst, een ontbrekende waarde en een echte `-`. Dat onderscheid is belangrijk: leeg betekent niet automatisch nul en een bronwaarde `-` mag niet stil worden aangepast.

In [ ]:
SMALL_DOCUMENT = '''# Premieoverzicht 2026

Onderstaande percentages zijn fictief en alleen bedoeld om de verwerking te demonstreren.

| Regeling | OP 2026 | RP 2026 | Totaal 2026 | Toelichting |
|---|---:|---:|---:|---|
| Regeling A | 22,00% | 3,93% | 25,93% | Standaardregeling |
| Regeling B | 22,00% | 3,73% | 25,73% | Afwijkende risicodekking |
| Regeling C | 24,00% |  | 24,00% | RP is niet vermeld |
| Regeling D | 21,00% | - | 21,00% | De bron bevat letterlijk een streepje |

Een ontbrekende waarde en een streepje behouden ieder hun eigen betekenis.'''

display(Markdown(SMALL_DOCUMENT))

## 2. Dunne Markdown-adapter

Deze adapter vindt één gangbare pipe-tabel in een document en zet die om naar het canonieke contract `columns + rows + context + spans`. Hij is bewust niet de universele parser: complexe HTML-tabellen, `rowspan`/`colspan` en Document Intelligence-cellen worden later via andere adapters aangeleverd.

In [ ]:
def split_row(line):
    return [cell.strip() for cell in line.strip().strip('|').split('|')]

def is_separator(line):
    cells = split_row(line)
    return len(cells) > 1 and all(
        re.fullmatch(r':?-{3,}:?', cell.replace(' ', ''))
        for cell in cells
    )

def markdown_to_canonical(document, source, ordinal=0):
    lines = document.splitlines()
    offsets, cursor = [], 0
    for line in lines:
        offsets.append(cursor)
        cursor += len(line) + 1

    start = next(
        index for index in range(len(lines) - 1)
        if '|' in lines[index] and is_separator(lines[index + 1])
    )
    end = start + 2
    while end < len(lines) and '|' in lines[end]:
        end += 1

    columns = split_row(lines[start])
    rows = []
    for line_index in range(start + 2, end):
        values = split_row(lines[line_index])
        if len(values) != len(columns):
            raise ValueError(f'Ongeldig kolomaantal op regel {line_index + 1}')

        cells, search_from = [], 0
        for value in values:
            local_start = lines[line_index].find(value, search_from)
            local_start = max(local_start, search_from)
            search_from = local_start + len(value)
            cells.append({
                'raw': value,
                'missing': value == '',
                'span': {
                    'start': offsets[line_index] + local_start,
                    'end': offsets[line_index] + local_start + len(value),
                },
            })

        rows.append({
            'header': values[0] or None,
            'span': {
                'start': offsets[line_index],
                'end': offsets[line_index] + len(lines[line_index]),
            },
            'cells': cells,
        })

    heading = next(
        (line.lstrip('#').strip() for line in reversed(lines[:start]) if line.startswith('#')),
        None,
    )
    before = '\n'.join(line for line in lines[:start] if not line.startswith('#')).strip()
    after = '\n'.join(lines[end:]).strip()
    raw = '\n'.join(lines[start:end])

    return {
        'source': source,
        'ordinal': ordinal,
        'caption': heading,
        'description': 'Tabel met waarden per regeling en jaar.',
        'context': {'before': before, 'after': after},
        'span': {'start': offsets[start], 'end': offsets[end - 1] + len(lines[end - 1])},
        'raw': raw,
        'columns': columns,
        'rows': rows,
    }

small_table = markdown_to_canonical(
    SMALL_DOCUMENT,
    {'uri': 'sharepoint://demo/premies', 'version': 'v1'},
)
show_json({key: small_table[key] for key in ('caption', 'context', 'span', 'columns')})

## 3. Generieke node- en chunkkern

Deze functie kent de bronvorm niet. Iedere adapter levert hetzelfde canonieke model aan. De kern maakt stabiele IDs, bewaart metadata en groepeert alleen volledige rijen. `max_size` gebruikt hier tekens voor een zichtbare demo; productie kan dezelfde functie een tokenizer als `measure` meegeven.

In [ ]:
def table_to_nodes(table, max_size=1200, measure=len):
    source = table['source']
    seed = f"{source['uri']}|{source.get('version')}|{table.get('ordinal', 0)}"
    table_id = table.get('id') or f"tbl-{sha1(seed.encode()).hexdigest()[:16]}"
    columns = [column if isinstance(column, dict) else {'name': str(column)} for column in table['columns']]
    label = lambda column: ' > '.join(map(str, column.get('path') or [column['name']]))
    node_id = lambda *parts: ':'.join([table_id, *map(str, parts)])

    rows = []
    for row_index, row in enumerate(table['rows']):
        if len(row['cells']) != len(columns):
            raise ValueError(f'Ongeldig kolomaantal in rij {row_index}')
        cells = []
        for column_index, cell in enumerate(row['cells']):
            cell_id = node_id('r', row_index, 'c', column_index)
            text = '' if cell.get('raw') is None else str(cell.get('raw'))
            cells.append({
                'id': cell_id,
                'type': 'cell',
                'row_index': row_index,
                'row_header': row.get('header'),
                'column_index': column_index,
                'column_name': label(columns[column_index]),
                'raw': cell.get('raw'),
                'missing': cell.get('missing', text == ''),
                'source_span': cell.get('span'),
                'rowspan': cell.get('rowspan', 1),
                'colspan': cell.get('colspan', 1),
                'items': [{
                    'id': node_id('r', row_index, 'c', column_index, 's', 0),
                    'type': 'sentence',
                    'text': text,
                }],
            })
        rows.append({
            'id': node_id('r', row_index),
            'type': 'table_row',
            'row_index': row_index,
            'row_header': row.get('header'),
            'source_span': row.get('span'),
            'items': cells,
        })

    def escape(value):
        return ' '.join(str(value).splitlines()).replace('\\', '\\\\').replace('|', '\\|')

    def display_cell(cell):
        text = ' '.join(item['text'] for item in cell['items']).strip()
        return text if text else '—'

    prefix = '\n\n'.join(filter(None, [
        table.get('caption'),
        table.get('description'),
        table.get('context', {}).get('before'),
    ]))
    suffix = table.get('context', {}).get('after', '')

    def render(group):
        headers = [escape(label(column)) for column in columns]
        markdown = [
            '| ' + ' | '.join(headers) + ' |',
            '| ' + ' | '.join(['---'] * len(headers)) + ' |',
            *[
                '| ' + ' | '.join(escape(display_cell(cell)) for cell in row['items']) + ' |'
                for row in group
            ],
        ]
        return '\n\n'.join(filter(None, [prefix, '\n'.join(markdown), suffix]))

    groups, current = [], []
    for row in rows:
        if current and measure(render(current + [row])) > max_size:
            groups.append(current)
            current = []
        current.append(row)
    groups.append(current)

    chunks = [{
        'chunk_index': index,
        'chunk': render(group),
        'content_type': 'table',
        'table_id': table_id,
        'row_start': group[0]['row_index'] if group else None,
        'row_end': group[-1]['row_index'] if group else None,
        'node_ids': [
            item['id'] for row in group for cell in row['items'] for item in cell['items']
        ],
        'source': source,
        'oversize_row': bool(group) and measure(render(group)) > max_size,
    } for index, group in enumerate(groups)]

    return {
        'id': table_id,
        'type': 'table',
        'caption': table.get('caption'),
        'description': table.get('description'),
        'context': table.get('context', {}),
        'source': source,
        'source_span': table.get('span'),
        'columns': columns,
        'items': rows,
        'chunks': chunks,
    }

## 4. Kleine tabel: nodes, metadata en frontendweergave

In [ ]:
small_result = table_to_nodes(small_table, max_size=2000)

print('table_id:', small_result['id'])
print('rows:', len(small_result['items']))
print('chunks:', len(small_result['chunks']))
print('node_ids:', len(small_result['chunks'][0]['node_ids']))
display(Markdown(small_result['chunks'][0]['chunk']))

In [ ]:
example_cell = small_result['items'][2]['items'][2]
show_json(example_cell)

## 5. Grote tabel: uitsluitend splitsen tussen volledige rijen

De volgende cel genereert synthetische Markdown met 35 rijen. Een klein chunkbudget maakt het splitsgedrag zichtbaar.

In [ ]:
def make_large_document(row_count=35):
    rows = [
        f'| Regeling {index:02d} | {20 + index % 5},00% | {2 + index % 3},50% | {22 + index % 7},50% | Voorbeeldregel {index:02d} |'
        for index in range(1, row_count + 1)
    ]
    return '\n'.join([
        '# Groot premieoverzicht',
        '',
        'Deze tabel demonstreert row-safe chunking.',
        '',
        '| Regeling | OP | RP | Totaal | Toelichting |',
        '|---|---:|---:|---:|---|',
        *rows,
        '',
        'De kolomkoppen moeten in iedere chunk terugkomen.',
    ])

LARGE_DOCUMENT = make_large_document()
large_table = markdown_to_canonical(
    LARGE_DOCUMENT,
    {'uri': 'sharepoint://demo/grote-tabel', 'version': 'v1'},
)
large_result = table_to_nodes(large_table, max_size=800)

summary = [{
    'chunk': chunk['chunk_index'],
    'rows': f"{chunk['row_start']}–{chunk['row_end']}",
    'characters': len(chunk['chunk']),
    'nodes': len(chunk['node_ids']),
    'oversize_row': chunk['oversize_row'],
} for chunk in large_result['chunks']]
show_json(summary)

In [ ]:
print('Eerste grote-table-chunk:')
display(Markdown(large_result['chunks'][0]['chunk']))

print('Laatste grote-table-chunk:')
display(Markdown(large_result['chunks'][-1]['chunk']))

## 6. Citation-click simuleren

De backend hoeft voor de frontend alleen de chunk te vinden waarvan `node_ids` de aangehaalde node bevat. De chunk zelf is reeds leesbare Markdown.

In [ ]:
def citation_chunk(table_result, node_id):
    return next(chunk for chunk in table_result['chunks'] if node_id in chunk['node_ids'])

citation_node_id = large_result['items'][17]['items'][3]['items'][0]['id']
selected_chunk = citation_chunk(large_result, citation_node_id)

print('Geklikte node:', citation_node_id)
print('Getoonde rijen:', selected_chunk['row_start'], 'tot', selected_chunk['row_end'])
display(Markdown(selected_chunk['chunk']))

## 7. Automatische controles

In [ ]:
all_node_ids = [
    item['id']
    for row in large_result['items']
    for cell in row['items']
    for item in cell['items']
]
chunk_node_ids = [node_id for chunk in large_result['chunks'] for node_id in chunk['node_ids']]
headers = '| Regeling | OP | RP | Totaal | Toelichting |'

assert len(all_node_ids) == len(set(all_node_ids)), 'Node IDs zijn niet uniek'
assert chunk_node_ids == all_node_ids, 'Rijen of nodes ontbreken of zijn dubbel gechunkt'
assert all(headers in chunk['chunk'] for chunk in large_result['chunks']), 'Headers ontbreken'
assert citation_node_id in selected_chunk['node_ids'], 'Citation is niet teruggevonden'
assert small_result['items'][2]['items'][2]['missing'] is True, 'Lege waarde is niet behouden'
assert small_result['items'][3]['items'][2]['raw'] == '-', 'Bronwaarde - is veranderd'

print('NOTEBOOK_CHECKS=PASS')

## Conclusie en vervolgstappen

**Wat nu aantoonbaar werkt in de spike**

- Canonieke tabel → traceerbare table/row/cell/sentence-nodes.
- Stabiele IDs, bronspans, rij- en kolommetadata.
- Lege waarden blijven onderscheiden van `-` en van nul.
- Grote tabellen splitsen alleen tussen volledige rijen.
- Iedere chunk bevat opnieuw de headers en is direct als Markdown toonbaar.
- Een citation-node resolveert naar één toonbare chunk.

**Nog niet bewezen / buiten deze spike**

1. SharePoint-HTML en Document Intelligence moeten adapters krijgen naar hetzelfde canonieke contract.
2. De tabelbeschrijving is hier statisch; de productiepipeline moet die begrensd met omliggende context genereren.
3. `max_size` moet in productie met de werkelijke tokenizer worden gemeten.
4. `node_ids`, spans en Markdown-chunk moeten door de indexering tot aan de bestaande retrieval/citationketen worden doorgegeven.
5. De React citation-weergave moet Markdown ondersteunen of een veilige fallback gebruiken.

**Voorgestelde eerstvolgende implementatiestap:** integreer eerst deze verticale slice voor één genormaliseerde Markdown-tabel in `DocumentPreparationPipeline`; voeg daarna pas de overige bronadapters en regressietabellen toe.